In [2]:
import pandas as pd
import json
import sqlite3

csv_df = pd.read_csv("users.csv")

print("Top 5 Records")
print(csv_df.head()) 

Top 5 Records
  user_id           name  age gender       city registration_date
0   U0001  Vihaan Sharma   35  Other     Jaipur        2022-09-08
1   U0002      Sai Reddy   30  Other  Hyderabad        2023-11-24
2   U0003   Aarohi Gupta   37  Other     Indore        2022-02-02
3   U0004    Aarav Gupta   44   Male    Kolkata        2023-06-02
4   U0005    Sara Sharma   30  Other    Chennai        2024-01-04


In [3]:
print("\nDataset Information")
print(csv_df.info())


Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            200 non-null    object
 1   name               200 non-null    object
 2   age                200 non-null    int64 
 3   gender             200 non-null    object
 4   city               200 non-null    object
 5   registration_date  200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB
None


In [4]:
print("\nData Types")
print(csv_df.dtypes)


Data Types
user_id              object
name                 object
age                   int64
gender               object
city                 object
registration_date    object
dtype: object


In [5]:
print("\nMissing Values")
print(csv_df.isnull().sum())


Missing Values
user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
dtype: int64


In [6]:
print("Duplicate Records =", csv_df.duplicated().sum())

Duplicate Records = 0


In [7]:
invalid_gender = csv_df[
    ~csv_df["gender"].isin(["Male","Female","Other"])
]

print(invalid_gender)

Empty DataFrame
Columns: [user_id, name, age, gender, city, registration_date]
Index: []


In [9]:
csv_df.to_json(
    "user_dataset.json",
    orient="records",
    indent=4
)

with open("user_dataset.json","r") as file:
    data = json.load(file)

json_df = pd.DataFrame(data)

print(json_df.head())

  user_id           name  age gender       city registration_date
0   U0001  Vihaan Sharma   35  Other     Jaipur        2022-09-08
1   U0002      Sai Reddy   30  Other  Hyderabad        2023-11-24
2   U0003   Aarohi Gupta   37  Other     Indore        2022-02-02
3   U0004    Aarav Gupta   44   Male    Kolkata        2023-06-02
4   U0005    Sara Sharma   30  Other    Chennai        2024-01-04


In [10]:
conn = sqlite3.connect("users.db")

csv_df.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

200

In [13]:
sql_df = pd.read_sql(
    "SELECT * FROM users",
    conn
)

print(sql_df.head())

  user_id           name  age gender       city registration_date
0   U0001  Vihaan Sharma   35  Other     Jaipur        2022-09-08
1   U0002      Sai Reddy   30  Other  Hyderabad        2023-11-24
2   U0003   Aarohi Gupta   37  Other     Indore        2022-02-02
3   U0004    Aarav Gupta   44   Male    Kolkata        2023-06-02
4   U0005    Sara Sharma   30  Other    Chennai        2024-01-04


In [15]:
print(csv_df.isnull().sum())

user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
dtype: int64


In [19]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='mean')

csv_df[['age']] = num_imputer.fit_transform(csv_df[['age']])

print(csv_df['age'].isnull().sum())

0


In [21]:
cat_imputer = SimpleImputer(strategy='most_frequent')

categorical_cols = ['gender','city']

csv_df[categorical_cols] = cat_imputer.fit_transform(csv_df[categorical_cols])

print(csv_df[categorical_cols].isnull().sum())

gender    0
city      0
dtype: int64


In [23]:
from sklearn.impute import KNNImputer

knn_df = csv_df.copy()

knn_imputer = KNNImputer(n_neighbors=3)

knn_df[['age']] = knn_imputer.fit_transform(knn_df[['age']])

print(knn_df.head())

  user_id           name   age gender       city registration_date
0   U0001  Vihaan Sharma  35.0  Other     Jaipur        2022-09-08
1   U0002      Sai Reddy  30.0  Other  Hyderabad        2023-11-24
2   U0003   Aarohi Gupta  37.0  Other     Indore        2022-02-02
3   U0004    Aarav Gupta  44.0   Male    Kolkata        2023-06-02
4   U0005    Sara Sharma  30.0  Other    Chennai        2024-01-04


In [27]:
csv_df['registration_date'] = pd.to_datetime(
    csv_df['registration_date'],
    errors='coerce'
)

print(csv_df['registration_date'].dtype)

datetime64[ns]


In [28]:
print("Before:",csv_df.shape)

csv_df = csv_df.drop_duplicates()

print("After:",csv_df.shape)

Before: (200, 6)
After: (200, 6)


In [30]:
from scipy.stats import zscore

df_z = csv_df.copy()

df_z['z_score'] = zscore(df_z['age'])

outliers_z = df_z[
    (df_z['z_score'] > 3) |
    (df_z['z_score'] < -3)
]

print(outliers_z)

Empty DataFrame
Columns: [user_id, name, age, gender, city, registration_date, z_score]
Index: []


In [31]:
df_z_clean = df_z[
    (df_z['z_score'] <= 3) &
    (df_z['z_score'] >= -3)
]

print(df_z_clean.shape)

(200, 7)


In [33]:
Q1 = csv_df['age'].quantile(0.25)
Q3 = csv_df['age'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

print(lower, upper)

12.125 49.125


In [35]:
outliers_iqr = csv_df[
    (csv_df['age'] < lower) |
    (csv_df['age'] > upper)
]

print(outliers_iqr)

    user_id           name   age  gender     city registration_date
113   U0114      Sai Singh  51.0    Male  Chennai        2022-05-31
179   U0180  Reyansh Verma  53.0  Female   Bhopal        2022-04-27


In [36]:
df_iqr_clean = csv_df[
    (csv_df['age'] >= lower) &
    (csv_df['age'] <= upper)
]

print(df_iqr_clean.shape)

(198, 6)


In [38]:
from scipy.stats.mstats import winsorize

csv_df['age_winsor'] = winsorize(
    csv_df['age'],
    limits=[0.05,0.05]
)

print(csv_df[['age','age_winsor']].head())

    age  age_winsor
0  35.0        35.0
1  30.0        30.0
2  37.0        37.0
3  44.0        44.0
4  30.0        30.0


In [40]:
print("Original Shape :",csv_df.shape)
print("Z-score Shape :",df_z_clean.shape)
print("IQR Shape :",df_iqr_clean.shape)

Original Shape : (200, 7)
Z-score Shape : (200, 7)
IQR Shape : (198, 6)


In [43]:
# Convert to Date Age Month Year
csv_df['registration_date'] = pd.to_datetime(csv_df['registration_date'])

csv_df['Day'] = csv_df['registration_date'].dt.day
csv_df['Month'] = csv_df['registration_date'].dt.month
csv_df['Year'] = csv_df['registration_date'].dt.year

print(csv_df[['registration_date','Day','Month','Year']].head())

  registration_date  Day  Month  Year
0        2022-09-08    8      9  2022
1        2023-11-24   24     11  2023
2        2022-02-02    2      2  2022
3        2023-06-02    2      6  2023
4        2024-01-04    4      1  2024


In [44]:
# Lable Encoding (binary)
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv("users.csv")

csv_df['is_active'] = ['Yes','No'] * 100

le = LabelEncoder()

csv_df['is_active_encoded'] = le.fit_transform(csv_df['is_active'])

print(csv_df[['is_active','is_active_encoded']].head())

  is_active  is_active_encoded
0       Yes                  1
1        No                  0
2       Yes                  1
3        No                  0
4       Yes                  1


In [45]:
# One Hot - Encoding (numerical)
city_encoded = pd.get_dummies(
    csv_df,
    columns=['city'],
    drop_first=True
)

print(city_encoded.head())

  user_id           name   age gender registration_date  age_winsor  Day  \
0   U0001  Vihaan Sharma  35.0  Other        2022-09-08        35.0    8   
1   U0002      Sai Reddy  30.0  Other        2023-11-24        30.0   24   
2   U0003   Aarohi Gupta  37.0  Other        2022-02-02        37.0    2   
3   U0004    Aarav Gupta  44.0   Male        2023-06-02        44.0    2   
4   U0005    Sara Sharma  30.0  Other        2024-01-04        30.0    4   

   Month  Year is_active  ...  city_Kolkata  city_Lucknow  city_Mumbai  \
0      9  2022       Yes  ...         False         False        False   
1     11  2023        No  ...         False         False        False   
2      2  2022       Yes  ...         False         False        False   
3      6  2023        No  ...          True         False        False   
4      1  2024       Yes  ...         False         False        False   

   city_Nagpur  city_Patna  city_Pune  city_Surat  city_Thane  city_Vadodara  \
0        False    

In [46]:
gender_encoded = pd.get_dummies(
    csv_df,
    columns=['gender'],
    drop_first=True
)

print(gender_encoded.head())

  user_id           name   age       city registration_date  age_winsor  Day  \
0   U0001  Vihaan Sharma  35.0     Jaipur        2022-09-08        35.0    8   
1   U0002      Sai Reddy  30.0  Hyderabad        2023-11-24        30.0   24   
2   U0003   Aarohi Gupta  37.0     Indore        2022-02-02        37.0    2   
3   U0004    Aarav Gupta  44.0    Kolkata        2023-06-02        44.0    2   
4   U0005    Sara Sharma  30.0    Chennai        2024-01-04        30.0    4   

   Month  Year is_active  is_active_encoded  gender_Male  gender_Other  
0      9  2022       Yes                  1        False          True  
1     11  2023        No                  0        False          True  
2      2  2022       Yes                  1        False          True  
3      6  2023        No                  0         True         False  
4      1  2024       Yes                  1        False          True  


In [49]:
# Ordinal Encoding(categorical)

csv_df['Membership'] = [
    'Silver','Gold','Platinum','Silver'
] * 50

from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(
    categories=[
        ['Silver','Gold','Platinum']
    ]
)

csv_df[['Membership']] = oe.fit_transform(
    csv_df[['Membership']]
)

print(csv_df[['Membership']].head())

   Membership
0         0.0
1         1.0
2         2.0
3         0.0
4         0.0


In [50]:
# Create Age Groups
import pandas as pd

csv_df['Age_Group'] = pd.cut(
    csv_df['age'],
    bins=[0, 25, 40, 100],
    labels=['Low', 'Medium', 'High']
)

print(csv_df[['age', 'Age_Group']].head(10))

    age Age_Group
0  35.0    Medium
1  30.0    Medium
2  37.0    Medium
3  44.0      High
4  30.0    Medium
5  30.0    Medium
6  44.0      High
7  38.0    Medium
8  28.0    Medium
9  36.0    Medium


In [54]:
import numpy as np

csv_df['spending'] = np.random.randint(1000,10000,size=len(csv_df))
csv_df['Spending_Group'] = pd.cut(
    csv_df['spending'],
    bins=[0, 1000, 5000, 10000],
    labels=['Low', 'Medium', 'High']
)

print(csv_df[['spending','Spending_Group']].head())

   spending Spending_Group
0      9016           High
1      6910           High
2      4039         Medium
3      8983           High
4      9480           High


In [55]:
# Log Transformation
import numpy as np

csv_df['Age_Log'] = np.log1p(csv_df['age'])

print(csv_df[['age', 'Age_Log']].head())

    age   Age_Log
0  35.0  3.583519
1  30.0  3.433987
2  37.0  3.637586
3  44.0  3.806662
4  30.0  3.433987


In [56]:
# Square Root Transformation
import numpy as np

csv_df['Age_Sqrt'] = np.sqrt(csv_df['age'])

print(csv_df[['age', 'Age_Sqrt']].head())

    age  Age_Sqrt
0  35.0  5.916080
1  30.0  5.477226
2  37.0  6.082763
3  44.0  6.633250
4  30.0  5.477226


In [57]:
#Apply StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

csv_df['Age_Standardized'] = scaler.fit_transform(csv_df[['age']])

print(csv_df[['age','Age_Standardized']].head())

    age  Age_Standardized
0  35.0          0.515666
1  30.0         -0.173727
2  37.0          0.791424
3  44.0          1.756575
4  30.0         -0.173727


In [58]:
# Min-Max Standarscaler
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

csv_df['Age_Normalized'] = scaler.fit_transform(csv_df[['age']])

print(csv_df[['age','Age_Normalized']].head())


    age  Age_Normalized
0  35.0        0.485714
1  30.0        0.342857
2  37.0        0.542857
3  44.0        0.742857
4  30.0        0.342857


In [59]:
print("Original Age Statistics")
print(csv_df['age'].describe())

print("Standardized Age Statistics")
print(csv_df['Age_Standardized'].describe())

print("Normalized Age Statistics")
print(csv_df['Age_Normalized'].describe())

Original Age Statistics
count    200.000000
mean      31.260000
std        7.270951
min       18.000000
25%       26.000000
50%       31.500000
75%       35.250000
max       53.000000
Name: age, dtype: float64
Standardized Age Statistics
count    2.000000e+02
mean    -1.776357e-16
std      1.002509e+00
min     -1.828272e+00
25%     -7.252420e-01
50%      3.309089e-02
75%      5.501361e-01
max      2.997483e+00
Name: Age_Standardized, dtype: float64
Normalized Age Statistics
count    200.000000
mean       0.378857
std        0.207741
min        0.000000
25%        0.228571
50%        0.385714
75%        0.492857
max        1.000000
Name: Age_Normalized, dtype: float64


In [61]:
# Feature Con
import pandas as pd

purchase_df = pd.DataFrame({
    'user_id':['U0001','U0001','U0002','U0002','U0003'],
    'purchase_date':['2024-01-10','2024-02-15','2024-01-20','2024-03-05','2024-02-25'],
    'category':['Electronics','Clothing','Grocery','Electronics','Clothing'],
    'amount':[5000,2000,1500,8000,3000]
})

# Average Month Sales

purchase_df['purchase_date'] = pd.to_datetime(purchase_df['purchase_date'])

print(purchase_df)
purchase_df['month'] = purchase_df['purchase_date'].dt.month

avg_monthly_spend = purchase_df.groupby(
    ['user_id','month']
)['amount'].mean().reset_index()

print(avg_monthly_spend)

  user_id purchase_date     category  amount
0   U0001    2024-01-10  Electronics    5000
1   U0001    2024-02-15     Clothing    2000
2   U0002    2024-01-20      Grocery    1500
3   U0002    2024-03-05  Electronics    8000
4   U0003    2024-02-25     Clothing    3000
  user_id  month  amount
0   U0001      1  5000.0
1   U0001      2  2000.0
2   U0002      1  1500.0
3   U0002      3  8000.0
4   U0003      2  3000.0


In [62]:
# Frequencey Purchase
purchase_frequency = purchase_df.groupby(
    'user_id'
).size().reset_index(name='Purchase_Frequency')

print(purchase_frequency)

  user_id  Purchase_Frequency
0   U0001                   2
1   U0002                   2
2   U0003                   1


In [63]:
# Last Purchase
today = pd.Timestamp('2024-12-31')

last_purchase = purchase_df.groupby(
    'user_id'
)['purchase_date'].max().reset_index()

last_purchase['Days_Since_Last_Purchase'] = (
    today - last_purchase['purchase_date']
).dt.days

print(last_purchase)

  user_id purchase_date  Days_Since_Last_Purchase
0   U0001    2024-02-15                       320
1   U0002    2024-03-05                       301
2   U0003    2024-02-25                       310


In [64]:
# Category-wise Total Expenditure
category_spend = purchase_df.groupby(
    'category'
)['amount'].sum().reset_index()

print(category_spend)

      category  amount
0     Clothing    5000
1  Electronics   13000
2      Grocery    1500


In [65]:
# Merge All Cleaned & Engineered Data
final_df = csv_df.copy()

print(final_df.shape)
print(final_df.head())

(200, 20)
  user_id           name   age gender       city registration_date  \
0   U0001  Vihaan Sharma  35.0  Other     Jaipur        2022-09-08   
1   U0002      Sai Reddy  30.0  Other  Hyderabad        2023-11-24   
2   U0003   Aarohi Gupta  37.0  Other     Indore        2022-02-02   
3   U0004    Aarav Gupta  44.0   Male    Kolkata        2023-06-02   
4   U0005    Sara Sharma  30.0  Other    Chennai        2024-01-04   

   age_winsor  Day  Month  Year is_active  is_active_encoded  Membership  \
0        35.0    8      9  2022       Yes                  1         0.0   
1        30.0   24     11  2023        No                  0         1.0   
2        37.0    2      2  2022       Yes                  1         2.0   
3        44.0    2      6  2023        No                  0         0.0   
4        30.0    4      1  2024       Yes                  1         0.0   

  Age_Group  spending Spending_Group   Age_Log  Age_Sqrt  Age_Standardized  \
0    Medium      9016           Hi

In [66]:
before_records = 200        
after_records = len(final_df)

print("Records Before Cleaning :", before_records)
print("Records After Cleaning  :", after_records)

Records Before Cleaning : 200
Records After Cleaning  : 200


In [67]:
# Number of Features Created
original_features = 6
final_features = final_df.shape[1]

created_features = final_features - original_features

print("Features Created :", created_features)

Features Created : 14


In [71]:
# Missing Values Summary
# Before
missing_before = csv_df.isnull().sum()
print("\n missing before",missing_before)

# After

missing_after = final_df.isnull().sum()
print("\n missing after",missing_after)

# Summary

summary_missing = pd.DataFrame({
    'Before': csv_df.isnull().sum(),
    'After': final_df.isnull().sum()
})

print("\n summary",summary_missing)



 missing before user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
age_winsor           0
Day                  0
Month                0
Year                 0
is_active            0
is_active_encoded    0
Membership           0
Age_Group            0
spending             0
Spending_Group       0
Age_Log              0
Age_Sqrt             0
Age_Standardized     0
Age_Normalized       0
dtype: int64

 missing after user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
age_winsor           0
Day                  0
Month                0
Year                 0
is_active            0
is_active_encoded    0
Membership           0
Age_Group            0
spending             0
Spending_Group       0
Age_Log              0
Age_Sqrt             0
Age_Standardized     0
Age_Normalized       0
dtype: int64

 summary            

In [72]:
# Outlier Count Before vs After
Q1 = csv_df['age'].quantile(0.25)
Q3 = csv_df['age'].quantile(0.75)

IQR = Q3 - Q1

outliers_before = csv_df[
    (csv_df['age'] < Q1 - 1.5*IQR) |
    (csv_df['age'] > Q3 + 1.5*IQR)
]

print("Outliers Before :", len(outliers_before))

Outliers Before : 2


In [73]:
Q1 = final_df['age'].quantile(0.25)
Q3 = final_df['age'].quantile(0.75)

IQR = Q3 - Q1

outliers_after = final_df[
    (final_df['age'] < Q1 - 1.5*IQR) |
    (final_df['age'] > Q3 + 1.5*IQR)
]

print("Outliers After :", len(outliers_after))

Outliers After : 2


C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


In [74]:
report = pd.DataFrame({
    'Metric':[
        'Records Before Cleaning',
        'Records After Cleaning',
        'Original Features',
        'Features Created',
        'Total Final Features',
        'Missing Values Before',
        'Missing Values After',
        'Outliers Before',
        'Outliers After'
    ],
    'Value':[
        before_records,
        after_records,
        original_features,
        created_features,
        final_df.shape[1],
        csv_df.isnull().sum().sum(),
        final_df.isnull().sum().sum(),
        len(outliers_before),
        len(outliers_after)
    ]
})

print("\n Final Report",report)


 Final Report                     Metric  Value
0  Records Before Cleaning    200
1   Records After Cleaning    200
2        Original Features      6
3         Features Created     14
4     Total Final Features     20
5    Missing Values Before      0
6     Missing Values After      0
7          Outliers Before      2
8           Outliers After      2


In [80]:
pip install ydata-profiling

Note: you may need to restart the kernel to use updated packages.


In [81]:
# Generate EDA Report
from ydata_profiling import ProfileReport

profile = ProfileReport(
    final_df,
    title="Customer Dataset EDA Report",
    explorative=True
)

profile.to_file("EDA_Report.html")

print("EDA Report Saved Successfully!")

Summarize dataset:   0%|                                         | 0/25 [00:00<?, ?it/s, Describe variable: age_winsor]C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:867: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
Summarize dataset:   4%|█▌                                     | 1/25 [00:00<00:02,  8.52it/s, Describe variable: Year]C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:867: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
Summarize dataset:   4%|█                         | 1/25 [00:00<00:02,  8.52it/s, Describe va

EDA Report Saved Successfully!


In [82]:
#  Save Cleaned Dataset
final_df.to_csv(
    "Final_Cleaned_Dataset.csv",
    index=False
)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!
